# n6 · Softmax、交叉熵与分类器训练  Softmax, Cross-Entropy & Classifier

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 推导 **softmax** 的数值稳定公式，解释为何要减去行最大值（避免数值溢出 numerical overflow）。
2. 推导 **交叉熵**（cross-entropy）损失并说明其与最大似然估计（MLE）的关系。
3. 推导并实现**融合 softmax 交叉熵**（fused softmax cross-entropy）的反向传播：
   $$\frac{\partial L}{\partial \text{logits}} = \frac{\text{softmax}(\text{logits}) - \text{onehot}(y)}{B}$$
4. 解释为何上述梯度公式与 PyTorch 的 `torch.nn.functional.cross_entropy` **完全一致**。
5. 用 `cross_entropy` 函数训练一个多类分类器，并验证损失与梯度的正确性。

> 对应能力：**SH8**

## 概念讲解 Concepts

### 1. Softmax：从 logit 到概率

多类分类中，网络输出 **logits** $z \in \mathbb{R}^{B \times C}$（$B$ 为批大小，$C$ 为类别数）。
Softmax 将 logits 转化为概率分布：

$$\text{softmax}(z_i)_k = \frac{e^{z_{ik}}}{\sum_{j=1}^C e^{z_{ij}}}$$

---

### 2. 数值稳定的 Softmax Numerically Stable Softmax

直接计算 $e^{z_{ik}}$ 当 $z_{ik}$ 较大时（如 1000）会溢出（overflow）。
解决方法：减去每行最大值（不改变概率分布，因为分子分母同乘同一常数）：

$$\text{softmax}(z)_k = \frac{e^{z_k - \max_j z_j}}{\sum_j e^{z_j - \max_j z_j}}$$

```python
z = logits - logits.max(axis=1, keepdims=True)   # 数值稳定
exp = np.exp(z)
sm = exp / exp.sum(axis=1, keepdims=True)
```

---

### 3. 交叉熵损失 Cross-Entropy Loss

对批量 $B$ 个样本，真实标签 $y_i \in \{0, \ldots, C-1\}$：

$$L = -\frac{1}{B} \sum_{i=1}^B \log \text{softmax}(z_i)_{y_i}$$

这等价于**负对数似然**（negative log-likelihood，NLL）。只有正确类的概率出现在损失中。

---

### 4. 融合梯度 Fused Gradient（核心推导）

$$\boxed{\frac{\partial L}{\partial z_{ik}} = \frac{\text{sm}_{ik} - \mathbf{1}[k = y_i]}{B}}$$

推导思路：
1. $L = -\frac{1}{B} \sum_i \log \text{sm}_{i,y_i}$
2. 对 $z_{ik}$ 求导，经过 softmax 的 Jacobian 化简后得到上式。
3. **直觉**：梯度 = 预测概率 - 真实标签（one-hot），再除以批大小 $B$。

实现中用 `onehot` 矩阵直接减：

```python
grad = sm.copy()
grad[np.arange(B), targets] -= 1.0
grad /= B
logits.grad = logits.grad + grad * out.grad
```

---

### 5. 与 PyTorch 的等价性

```python
import torch.nn.functional as F
loss = F.cross_entropy(logits, targets)
```

PyTorch 内部使用 **log-sum-exp** 技巧（数值等价），梯度公式与上面完全相同。
本课实现与 `F.cross_entropy` 在数值上几乎完全一致（差异 < 1e-6 来自 `1e-12` 的保护项）。

---

### 6. 融合算子的优势 Why Fuse?

若分别实现 softmax 和 log，梯度需流经两个节点，计算量翻倍且数值更不稳定。
将 softmax 和 cross-entropy **融合**成一个 `cross_entropy` 函数，只有一个节点，梯度直接高效计算。

## 示例 Worked Example

In [ ]:
# ── nanograd-l6 完整实现（含 cross_entropy）────────────────────────────────
import numpy as np


def _unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad


class Tensor:
    def __init__(self, data, _children=(), _op=""):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad = self.grad + _unbroadcast(out.grad, self.data.shape)
            other.grad = other.grad + _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad = self.grad + _unbroadcast(other.data * out.grad, self.data.shape)
            other.grad = other.grad + _unbroadcast(self.data * out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other), "@")
        def _backward():
            self.grad = self.grad + out.grad @ other.data.T
            other.grad = other.grad + self.data.T @ out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), (self,), "sum")
        def _backward():
            self.grad = self.grad + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0.0, self.data), (self,), "relu")
        def _backward():
            self.grad = self.grad + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1.0

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if id(v) not in visited:
                visited.add(id(v))
                for child in v._prev: build(child)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)
        for v in reversed(topo): v._backward()

    def __repr__(self):
        return f"Tensor(shape={self.data.shape})"


class Module:
    def parameters(self): return []
    def zero_grad(self):
        for p in self.parameters():
            p.grad = np.zeros_like(p.data)


class Linear(Module):
    def __init__(self, nin, nout, seed=0):
        rng = np.random.default_rng(seed)
        self.w = Tensor(rng.normal(0, 1.0 / np.sqrt(nin), (nin, nout)))
        self.b = Tensor(np.zeros(nout))

    def __call__(self, x): return x @ self.w + self.b
    def parameters(self): return [self.w, self.b]


class SGD:
    def __init__(self, params, lr=0.1):
        self.params = list(params)
        self.lr = lr

    def zero_grad(self):
        for p in self.params: p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.params: p.data = p.data - self.lr * p.grad


def cross_entropy(logits, targets):
    """融合 softmax 交叉熵（Fused softmax cross-entropy).

    参数:
        logits: Tensor, 形状 (B, C)，B=批大小，C=类别数
        targets: int 数组，形状 (B,)，每个元素 in [0, C)

    返回:
        loss: 标量 Tensor

    梯度公式: dL/d_logits = (softmax(logits) - onehot(targets)) / B
    """
    # 数值稳定 softmax（减去行最大值）
    z = logits.data - logits.data.max(axis=1, keepdims=True)
    exp = np.exp(z)
    sm = exp / exp.sum(axis=1, keepdims=True)

    B = z.shape[0]
    targets = np.asarray(targets, dtype=int)

    # NLL 损失（加 1e-12 防止 log(0)）
    loss_val = float(-np.log(sm[np.arange(B), targets] + 1e-12).mean())
    out = Tensor(loss_val, (logits,), "ce")

    def _backward():
        # 融合梯度: (softmax - onehot) / B
        grad = sm.copy()
        grad[np.arange(B), targets] -= 1.0
        grad /= B
        logits.grad = logits.grad + grad * out.grad

    out._backward = _backward
    return out


print("cross_entropy 函数定义完毕（nanograd-l6 镜像）")


In [ ]:
# ── 数值稳定 softmax 演示 Numerical Stability Demo ──────────────────────────
print("=== 数值稳定性对比 Numerical Stability Comparison ===")
print()

# 不稳定版本（直接 exp）
def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

# 稳定版本（减最大值）
def softmax_stable(z):
    z2 = z - z.max(axis=1, keepdims=True)
    e = np.exp(z2)
    return e / e.sum(axis=1, keepdims=True)

# 极端 logits（会导致溢出）
z_extreme = np.array([[1000.0, 1001.0, 999.0]])

print("极端 logits:", z_extreme)
try:
    sm_naive = softmax_naive(z_extreme)
    print(f"  朴素 softmax: {sm_naive}  (可能含 nan/inf)")
except Exception as e:
    print(f"  朴素 softmax: 异常 {e}")

sm_stable = softmax_stable(z_extreme)
print(f"  稳定 softmax: {sm_stable.round(4)}")
print(f"  总和: {sm_stable.sum():.6f}  (应为 1.0)")
print()

# 正常 logits
z_normal = np.array([[2.0, 1.0, 0.1], [0.5, 1.5, 3.0]])
sm = softmax_stable(z_normal)
print("正常 logits 的 softmax:")
print(f"  z = \n{z_normal}")
print(f"  softmax(z) = \n{sm.round(4)}")
print(f"  每行之和: {sm.sum(axis=1)}")
assert np.allclose(sm.sum(axis=1), 1.0)
print("✓ softmax 输出合法概率分布（每行和为1）")


In [ ]:
# ── 交叉熵损失与融合梯度演示 Cross-Entropy and Fused Gradient Demo ───────────
np.random.seed(0)

B, C = 6, 4   # 批大小=6，类别数=4
logits_np = np.random.randn(B, C)
targets_np = np.array([0, 1, 2, 3, 1, 0])

logits_t = Tensor(logits_np.copy())
loss = cross_entropy(logits_t, targets_np)
loss.backward()

print(f"cross_entropy 演示: B={B}, C={C}")
print(f"  loss.data = {loss.data:.6f}")
print(f"  logits.grad.shape = {logits_t.grad.shape}  (期望 ({B},{C}))")
print()

# 手工验证梯度
z = logits_np - logits_np.max(axis=1, keepdims=True)
exp = np.exp(z)
sm = exp / exp.sum(axis=1, keepdims=True)
expected_grad = sm.copy()
expected_grad[np.arange(B), targets_np] -= 1.0
expected_grad /= B

max_err = np.abs(logits_t.grad - expected_grad).max()
print(f"  梯度与手工计算的误差 max|grad - expected| = {max_err:.2e}")
assert max_err < 1e-10, f"梯度不匹配: {max_err}"
print("✓ 融合梯度 (softmax - onehot)/B 正确")


In [ ]:
# ── 与 PyTorch F.cross_entropy 对比（Oracle 验证，torch 可选）────────────
# 若 torch 不可用，此单元格跳过（torch is optional oracle）
try:
    import torch as _torch
    import torch.nn.functional as _F
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False
    print("torch 不可用，跳过 PyTorch oracle 对比 (torch not installed)")

np.random.seed(42)
B, C = 8, 5
logits_np = np.random.randn(B, C)
targets_np = np.random.randint(0, C, size=B)

if _HAS_TORCH:
    # ── nanograd ────────────────────────────────────────────────────────────────
    logits_ng = Tensor(logits_np.copy())
    loss_ng = cross_entropy(logits_ng, targets_np)
    loss_ng.backward()

    # ── PyTorch ─────────────────────────────────────────────────────────────────
    logits_pt = _torch.tensor(logits_np, dtype=_torch.float64, requires_grad=True)
    targets_pt = _torch.tensor(targets_np, dtype=_torch.long)
    loss_pt = _F.cross_entropy(logits_pt, targets_pt)
    loss_pt.backward()

    loss_err  = abs(float(loss_ng.data) - float(loss_pt.item()))
    grad_err  = np.abs(logits_ng.grad - logits_pt.grad.numpy()).max()

    print("=== nanograd cross_entropy vs PyTorch F.cross_entropy ===")
    print(f"  nanograd  loss = {float(loss_ng.data):.8f}")
    print(f"  PyTorch   loss = {float(loss_pt.item()):.8f}")
    print(f"  loss 误差 = {loss_err:.2e}  (应 < 1e-5)")
    print()
    print(f"  grad 最大误差 = {grad_err:.2e}  (应 < 1e-5)")
    print()

    assert loss_err < 1e-5, f"loss 不一致: {loss_err}"
    assert grad_err < 1e-5, f"grad 不一致: {grad_err}"
    print("✓ 与 PyTorch F.cross_entropy 完全一致！")
    print("  这正是 PyTorch 内部的做法：融合 softmax 的数值稳定交叉熵。")


In [ ]:
# ── 完整分类器训练：3 类合成数据集 Full Classifier Training ─────────────────
np.random.seed(2025)

# 合成 3 类线性可分数据
C_cls = 3
N_per = 60
X_parts, Y_parts = [], []
for c in range(C_cls):
    angle = c * 2 * np.pi / C_cls
    center = np.array([np.cos(angle) * 2, np.sin(angle) * 2])
    X_parts.append(np.random.randn(N_per, 2) * 0.4 + center)
    Y_parts.append(np.full(N_per, c, dtype=int))

X_cls_np = np.vstack(X_parts)
Y_cls_np = np.hstack(Y_parts)

# 打乱顺序
perm = np.random.permutation(len(Y_cls_np))
X_cls_np = X_cls_np[perm]
Y_cls_np = Y_cls_np[perm]

# 模型：Linear(2, 3)（单层 softmax 分类器）
classifier = Linear(2, C_cls, seed=0)
optimizer = SGD(classifier.parameters(), lr=0.1)

X_t = Tensor(X_cls_np)

EPOCHS = 200
losses_cls = []

for epoch in range(EPOCHS):
    logits = classifier(X_t)
    loss = cross_entropy(logits, Y_cls_np)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses_cls.append(float(loss.data))

    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        # 计算准确率
        z = logits.data - logits.data.max(axis=1, keepdims=True)
        exp_ = np.exp(z)
        sm_ = exp_ / exp_.sum(axis=1, keepdims=True)
        preds_ = sm_.argmax(axis=1)
        acc_ = (preds_ == Y_cls_np).mean()
        print(f"  epoch {epoch:3d}  loss={loss.data:.4f}  acc={acc_:.3f}")

print()
# 最终准确率
logits_final = classifier(X_t)
z = logits_final.data - logits_final.data.max(axis=1, keepdims=True)
exp_ = np.exp(z)
sm_ = exp_ / exp_.sum(axis=1, keepdims=True)
acc_final = (sm_.argmax(axis=1) == Y_cls_np).mean()

print(f"最终准确率 Final accuracy: {acc_final:.3f}  (期望 > 0.90)")
assert acc_final > 0.90, f"准确率不足: {acc_final:.3f}"
assert losses_cls[-1] < losses_cls[0] * 0.2
print("✓ 分类器训练成功，损失下降 80% 以上，准确率达标")


## 动手 Exercise

In [ ]:
# ── 动手练习：双层分类网络 + 梯度校验 Two-Layer Classifier + Grad Check ────
# 任务：构建 Linear(2,16) → ReLU → Linear(16,3) 的网络
# 1. 训练 100 步，确认损失下降
# 2. 对第一个 logit 参数做梯度校验，确认数值梯度与 autograd 一致

np.random.seed(7)

# 复用上面的 3 类数据 X_cls_np, Y_cls_np
N_check = 20   # 用小批量做梯度校验
X_check_np = X_cls_np[:N_check]
Y_check_np = Y_cls_np[:N_check]


class TwoLayerClassifier(Module):
    def __init__(self, seed=0):
        self.l1 = Linear(2, 16, seed=seed)
        self.l2 = Linear(16, 3, seed=seed + 1)

    def __call__(self, x):
        return self.l2(self.l1(x).relu())

    def parameters(self):
        return self.l1.parameters() + self.l2.parameters()


net = TwoLayerClassifier(seed=0)
opt = SGD(net.parameters(), lr=0.05)

X_ck = Tensor(X_check_np)

for step in range(100):
    logits_ck = net(X_ck)
    loss_ck = cross_entropy(logits_ck, Y_check_np)
    opt.zero_grad()
    loss_ck.backward()
    opt.step()

print(f"100 步后损失 loss = {float(loss_ck.data):.4f}  (初始通常 ~1.1)")
assert float(loss_ck.data) < 1.0, "损失未下降！"
print("✓ 双层分类器损失下降正常")

# ── 梯度校验（对第一层第一个权重元素）────────────────────────────────────────
print()
print("=== 数值梯度校验 Numerical Gradient Check ===")

w0 = net.l1.w   # 形状 (2, 16)
h = 1e-4

def compute_loss(delta, idx):
    """对 w0[idx] 施加 delta 扰动后计算损失."""
    w0.data[idx] += delta
    lg = net(X_ck)
    ls = cross_entropy(lg, Y_check_np)
    val = float(ls.data)
    w0.data[idx] -= delta
    return val

# 对 w0[0,0] 做数值梯度
idx = (0, 0)
loss_plus  = compute_loss(+h, idx)
loss_minus = compute_loss(-h, idx)
nd = (loss_plus - loss_minus) / (2 * h)

# autograd 梯度
logits_ck = net(X_ck)
loss_ck = cross_entropy(logits_ck, Y_check_np)
opt.zero_grad()
loss_ck.backward()
ag = w0.grad[idx]

print(f"  w0[0,0] autograd  梯度: {ag:.8f}")
print(f"  w0[0,0] 数值差分  梯度: {nd:.8f}")
print(f"  误差: {abs(ag - nd):.2e}  (期望 < 1e-4)")
assert abs(ag - nd) < 1e-4, f"梯度不匹配: ag={ag:.6f}, nd={nd:.6f}"
print("✓ 数值梯度校验通过")


## 延伸阅读 Further Reading

1. **PyTorch `F.cross_entropy` 文档**：<https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html> — 含数学定义和 `reduction` 参数说明，与本课实现直接对应。
2. **Softmax 推导与数值稳定性**：<https://cs231n.github.io/linear-classify/#softmax-classifier> — CS231n 对 softmax 分类器的详细推导。
3. **交叉熵与 KL 散度**：$H(p,q) = H(p) + D_{KL}(p\|q)$ — 从信息论角度理解交叉熵损失：最小化交叉熵等价于最大化真实分布与预测分布的相似度。
4. **Karpathy «makemore» 系列**：<https://github.com/karpathy/makemore> — 字符级语言模型，使用 softmax 交叉熵作为训练目标，是本课的直接应用场景。
5. **Nielsen «Neural Networks and Deep Learning»** 第 3 章：交叉熵损失函数及其解决"学习缓慢"问题的原理。

## 关联练习 Related Assignments

```bash
python check.py nanograd-l6
```

> 实现 `cross_entropy(logits, targets)` 函数：融合 softmax、数值稳定交叉熵及梯度 `(softmax−onehot)/B`，与 PyTorch `F.cross_entropy` 一致。

> 能力标签：**SH8** · threshold ≥ 0.7